In [0]:
#####################           LOAD DATA FROM CSV FILE
#Previous CSV file was imported into a schema.table (mexico.portfolio_mexico)
pm=spark.sql("select * from mexico.portfolio_mexico")


In [0]:
#####################           INITIALIZE MODEL
 
import mlflow

# Define the model_uri
#/Users/adrian_esteva@epam.com/ML/insured_prediction_model
model_name = "m-insured_prediction_model"
model_version = "1"
#model_uri = f"models:/Users/adrian_esteva@epam.com/ML/m-insured_prediction_model" # Not detected
#model_uri = f"models:/m-insured_prediction_model"

# Upload the model
predict_udf = mlflow.pyfunc.spark_udf(spark, model_uri=model_uri)

In [0]:
#####################           INITIALIZE PREDICTION


from pyspark.sql.functions import struct

# I dont understand which columns should I use to the predictions, so I take these 3 for the exercise
columns_mexico = ["coverage_type","premium","sum_insured"]

# Generate prediction based on these 3 columns
predictions_df = pm.withColumn(
    "prediction", 
    predict_udf(struct(*columns_mexico))
)

# Import library related to LinearRegression
from pyspark.ml.regression import LinearRegression

data_mexico=LinearRegression(sum_insured="sum_insured",)


In [0]:
#####################           SAVE RESULTS ON A TABLE

# Table mexico.portfolio_mexico_prediction
(predictions_df.write
 .format("delta")
 .mode("append") 
 .saveAsTable("mexico.portfolio_mexico_prediction"))


In [0]:
#Everytime dataframe "pm" change with the new values, compare with the data saved in data_mexico
alert_insured=pm.join(pm, pm.coverage_type == data_mexico.coverage_type,"outer").filter(pm.sum_insured != data_mexico.sum_insured)

#On this point the alerts should be applied only for the different rows